# Miniproject template

This notebook provides the technical information for realizing your miniproject. For the general information see the [Miniproject slides](https://docs.google.com/presentation/d/1YEVQz4LuWjEf0bp1UInlDOeoY4H4G5eAISsCEr77Qbo/edit?usp=sharing).

This notebook has two main purposes:
- It presents the generic scene available for the miniproject and explain how to customize it to implement your own scenario. 
- It provides a template for your own miniproject, that you can implement and document directly in this notebook.

We recommend to first read and execute this entire notebook as it is, without attempting to modify it for your own scenario. This way you will be fully aware of all the available functionalities that you can use before starting to customize your own simulation. 

Then, you can start modifying this notebook to your own needs. You can always have access to the [original version of this document online](https://github.com/flowersteam/vivarium/tree/upf2026/notebooks/sessions/miniproject_template.ipynb) if you need it. When you start modifying this notebook for your miniproject, you can use *markdown cells* to add text describing your project, and *code cells* to implement you simulation (see [session 1](https://github.com/flowersteam/vivarium/tree/upf2026/notebooks/sessions/session_1.ipynb) for an explanation of the difference between markdown and code cells). The notebook you will deliver will have to be "self-contained", i.e. to clearly describe your scenario, how you implement it with code, and what the reader should observe in the simulation (see the [Miniproject slides](https://docs.google.com/presentation/d/1YEVQz4LuWjEf0bp1UInlDOeoY4H4G5eAISsCEr77Qbo/edit?usp=sharing) for more information on what to deliver).

When implementing your scenario you might encounter issues or limitations. For instance, you would have liked to implement fancy interactions between agents, but it turned out that this was not possible with the available functionalities, or you didn't figure out how to do it. In this case, we recommend that you simplify your scenario to make sure it can be implemented with the available functionalities, and to explain in the conclusion of your notebook what you would have like to do, what issues you encountered etc...

More genrally, we strongly recommend to structure the implementation of your scenario in an iterative way. Or in other words: first start simple, then add more complexity step by step. The most important is that you can deliver on time a miniproject that works. Therefore, you have interest to first work on a first and easy version of your scenario and save it as a backup once it works, before continuing for the next steps. This way you will ensure that you always have a working version you can deliver, whatever happens next. We provide guidelines on how to back up your work on retrieve previous versions later in this notebook.

If you experience trouble during your miniproject, feel free to contact us on Aula Global with a precise description of your problem and a copy of your notebook file. The notebook file you are currently reading is named `miniproject_template.ipynb` and is located on your computer in the folder indicated when you execute the following cell:

In [ ]:
pwd

As usual, let's start by connecting this notebook to the simulator:

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="miniproject")

## Available entities

We provide a generic scene containing **12 agents** (blue squares) and **32 objects** (green circles), that you can see on simulation map. You can freely customize these 44 entities in order to implement you own scenario, as explained below.

## Defining custom subtype labels

During the practical sessions, we were using *subtype labels* to distinguish between different categories of entities. The subtype labels were predefined for the purpose of the sessions and we used meaningful labels, such as `"obstacle"` for entities that agents need to avoid or `"resource"` for entities that agents want to forage for. These subtype labels were used for several purposes, e.g. enabling agents to selective sense specific entities with commands such as `agent.proximeters(sensed_entities=["obstacle"])`, or specifying which entities to spawn in the environment with command such as `controller.spawn.subtype = "resource"` (see [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb) and [session 4](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_4.ipynb)).

In your miniproject, however, you will want to define your own meaningful subtypes for categorizing the agents and objects according to the scenario you have in mind. Some of you might want to use subtypes for referring to e.g. `"prey"` or `"predator"` agents, while others might want to instead refer to `"bee"` agents and `"flower"` objects, or whatever you might have in mind. 

By default, the current scene provides 8 generic subtypes. The names of these generic subtypes can be accessed with:

In [ ]:
controller.subtypes

As an example, let's imagine you want to implement a classical scenario with prey and predators agents, as well as resource, obstacle and tree objects. For such a scenario you can define you own meaningful subtype labels with:

In [ ]:
controller.set_subtype_labels(['prey', 'predator', 'resource', 'obstacle', 'tree'])

Note that the choice of subtype labels is totally up to you, you can choose whatever labels that make sense for you scenario, e.g. `"bird"`, `"flower"`, `"bee"`, `"box"`, `"home"` ... whatever. The only limitation is that **you can only define a maximum of 8 subtype labels**. If you provide more than 8 subtypes in the command above, it will raise an error. 

Now that we have defined our example custom subtypes above with `controller.set_subtype_labels(['prey', 'predator', 'resource', 'obstacle', 'tree'])`, the list of available subtypes as been updated accordingly.  We can check it with:

In [ ]:
controller.subtypes

The custom subtypes we have define appear at the top of the list. Note that the remaining subtypes at the end of the list still correspond to the original generic subtypes (`"subtype_6"`, `"subtype_7"` and `"subtype_8"`). This is fine, but you will most likely prefer to only refer to your own custom subtypes in your code (as we will do in the examples below).

We strongly recommend to define your custom subtype labels at the start of your notebook, just after having connected the notebook to the simulation with `controller = VivariumController.start_session(scene_name="miniproject")`. **Define your custom subtypes only once in your notebook and do not change them later in your following code**.


To demonstrate all the functionalities that are available we will now go through a classical example of a prey-predator ecosystem. Your own scenario can for instance build upon this example to make it more interesting or more complex, but of course it can also be completely different. Be creative!

## Assigning subtypes and attributes to entities

Once you have defined your custom subtypes, you can assign them to the entites in the scene: the 12 available agents and the 32 available objects. 

As an example, let's consider the following scenario:

- Among the 12 agents:
    - 8 are considered as *prey* agents. They are assigned with the subtype `"prey"`, a diameter of 3, the color blue, and a maximum speed of 2.
    - 4 are considered as *predator* agents. They are assigned with the subtype `"predator"`, a diameter of 6, the color red, and a maximum speed of 1.
- Among the 32 object
    -  24 are considered as *resource* objects. They are assigned with the subtype `"resource"`, a diameter of 3 and the color green.
    -  7 are consided ad *obstacle* object. They are assigned with the subtype `"obstacle"`, a diameter of 8 and the color orange.
    -  1 is considered as a *tree* objects. It is assigned with the subtype `"tree"`, a diameter of 16 and the color brown.

For implementing this, we can use a `for` loop and the python indexing system we have started to see during the practical sessions and that we precise below. 

All agents in the scene are accessible through the `controller.agents` list, which contains the 12 agents available in this scene (the blue squares on the map) ; and all objects through the `controller.objects` list, which contains the 32 objects available in this scene (the green circles on the map). To access multiple elements in this list you can use the bracket indexing notation of Python. For instance, the 3 first agents in the `controller.agents` list can be accessed with:

In [ ]:
controller.agents[0:3]

The code cell above accesses agents from the `controller.agents` list with indexes from 0 (included) to 3 (excluded), therefore the three first agents in the list (the cell above prints them in a cryptic way, separated by comma). The "start" and "stop" indexes are indicated within the square brackets and separated by a column, i.e. `[0:3]` above. In Python, the convention is that the stop index is excluded from the resulting list. Therefore, `controller.agents[0:3]` refers to the three first agents of the list, i.e. `controller.agents[0]`, `controller.agents[1]` and `controller.agents[2]`. The same applies to the `controller.objects` list.

Knowing this, we can now assign subtypes and attributes to the entities in scene as follows. Let's start with the agents:


In [ ]:
# DEFINING 8 PREY AGENTS
# Iterate over the 8 first agents of the list, i.e. controller.agents[0:8]
# and set their subtype to "prey", their diameter to 3, their color to blue and their maximum speed to 2
for agent in controller.agents[0:8]:
    agent.subtype = "prey"
    agent.diameter = 3
    agent.color = "blue"
    agent.max_speed = 2


# DEFINING 4 PREDATOR AGENTS
# Iterate over the 4 next agents of the list, i.e. controller.agents[8:12]
# and set their subtype to "predator", their diameter to 6, their color to red and their maximum speed to 1
for agent in controller.agents[8:12]:
    agent.subtype = "predator"
    agent.diameter = 6
    agent.color = "red"
    agent.max_speed = 1


After executing the code cell above, you will see that the diameter and color of the agents have changed as expected. Their subtype and and maximum speed have also been changed but we cannot directly observe it on the map. If we want to double check the the above settings are effective we can e.g. execute:

In [ ]:
# Print the current subtype, diameter, color and max_speed attributes of all agents
for agent in controller.agents:
    print(agent.subtype, agent.diameter, agent.color, agent.max_speed)

The cell above displays the current subtype, diameter, color and maximum speed of each agent (one agent per row). As we can see, 8 agents have the subtype `prey` with the prey attributes we defined, and the 4 others have the subtype `predator` with the predator attributes we defined. All good.

Now let's assign the attributes of the 32 objects in the scene according to the scenario we have outlined above:

In [ ]:
# DEFINING 24 RESOURCE OBJECTS
# Iterate over the 24 first objects of the list, i.e. controller.objects[0:24]
# and set their subtype to "resource", their diameter to 3 and their color to green
for obj in controller.objects[0:24]:
    obj.subtype = "resource"
    obj.diameter = 3
    obj.color = "green"


# DEFINING 7 OBSTACLE OBJECTS
# Iterate over the 7 next objects of the list, i.e. controller.objects[24:31]
# and set their subtype to "obstacle", their diameter to 8 and their color to orange
for obj in controller.objects[24:31]:
    obj.subtype = "obstacle"
    obj.diameter = 8
    obj.color = "orange"

# DEFINING 1 TREE OBJECT
# Iterate over the last object of the list, i.e. controller.objects[31:32]
# and set its subtype to "tree", its diameter to 16 and its color to brown
for obj in controller.objects[31:32]:
    obj.subtype = "tree"
    obj.diameter = 16
    obj.color = "brown"

After executing the code cell above, you will see that the diameter and color of the objects have changed as expected.

In your own miniproject, you can use similar instructions to define different populations of agents or objects and change their attributes as you want, e.g. to make them easy to distinguish or to convey some meaning (for example deciding that an object will represent a tree and will be brown, with a larger diameter than a flower object which will be purple). In your code, make sure you use the same subtype labels as you have defined them earlier in the notebook with the `controller.set_subtype_labels([...])` command (otherwise it will raise an error, indicating the valid subtypes).

## Accessing specific entities

Above we assigned subtypes and attributes to agents and objects in batch, for instance assigning the subtype `"obstacle"`, a diameter of 8 and the color orange to 7 objects of the `controller.objects` list.

Let's say we now want the color of all object with the subtype `"obstacle"` to be yellow instead. For this we can write:

In [ ]:
for obj in controller.objects:  # Iterate over all objects of the list
    if obj.subtype == "obstacle":  # If the subtype of the object is "obstacle"
        obj.color = "yellow"  # Set the color of the object to yellow

All obstacle objects are now yellow.

What if we now want to assign the color purple to only 4 the 7 obstacle objects? When we have defined the `"obstacle"` subtype to objects above we used `for obj in controller.objects[24:31]:`, which means that the obstacle objects corresponds to indexes 24 (included) to 31 (excluded). From these 7 indices we can choose the 4 first one, i.e. `controller.objects[24:28]`, and assign them the color purple with:

In [ ]:
for obj in controller.objects[24:28]:  # Iterate over the 4 first objects of the list of obstacles
    obj.color = "purple"  # Set the color of the object to yellow

If we want to access a single entity, for example the only object with subtype `"tree"`, we can look at the index with used for it when we first set its attributes. It is index 31. Let's say we want to modify its diameter, we can write:

In [ ]:
tree = controller.objects[31]

Let's check it indeed has the subtype  `"tree"`:

In [ ]:
tree.subtype

And change its diameter:

In [ ]:
tree.diameter = 18

Now, among the 7 objects with subtype `"obstacle"`, 4 are purple and 3 are yellow. 

Now the tree object 

## Attaching behaviors to specific agent's subtypes

Let's continue to refine our example scenario. We will now implement the following:

- All agents (both prey and predator agents) are equipped with a behavior to avoid the obstacles and the tree.
- The prey agent are equipped with a behavior to forage for resources and to fear the predators
- The predator agents are equipped with a behavior to attack the preys.

We first define the required behaviors, similarly to what we have learned during the practical sessions:

In [ ]:
# Behavior that avoids entities with subtype "obstacle" or "tree"
def obstacle_avoidance(agent):
    left, right = agent.proximeters(sensed_entities=["obstacle", "tree"])
    left_motor = 1 - right
    right_motor = 1 - left
    return left_motor, right_motor  

# Behavior that attracts the agent toward entities with subtype "resource"
def foraging(agent):
    left, right = agent.proximeters(sensed_entities=["resource"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor

# Behavior that repulse the agent from entities with subtype "predator"
def fear(agent):
    left, right = agent.proximeters(sensed_entities=["predator"])
    left_motor = left
    right_motor = right
    return left_motor, right_motor

# Behavior that attracts the agent toward entities with subtype "prey"
def attack(agent):
    left, right = agent.proximeters(sensed_entities=["prey"])
    left_motor = right
    right_motor = left
    return left_motor, right_motor

And finally we assign these behaviors to the corresponding agent subtypes:

In [ ]:
# First detach all potential previous behaviors and routines from all agents
# No behavior or routine should be currently attached, but it is always safer
# to first remove them, e.g. in case you have to re-execute this cell later on
for agent in controller.agents:
    agent.detach_all_behaviors(stop_motors=True)
    agent.detach_all_routines()


# Attach behaviors to prey agents
for agent in controller.agents:  # Iterate over all agents
    if agent.subtype == "prey":  # If the agent has the subtype "prey"
        agent.attach_behavior(obstacle_avoidance)  # Attach the corresponding behaviors we specified in our scenario
        agent.attach_behavior(foraging)
        agent.attach_behavior(fear)


# Attach behaviors to predator agents
for agent in controller.agents:  # Iterate over all agents
    if agent.subtype == "predator":  # If the agent has the subtype "predator"
        agent.attach_behavior(obstacle_avoidance)  # Attach the corresponding behaviors we specified in our scenario
        agent.attach_behavior(attack)

Now our agents are in movement, each one executing the correct set of behaviors according to its subtype. Prey agents forage for resources and are afraid of predator agents, while predator agents chase preys, and all of them avoid the obstacles and the tree. The maximum speed of the preys is also twice faster than the one of the predators. This might be hard to observe on the map though, because the foraging behavior of preys make them push the resources, which slows them down. It will be more evident when will have activated the consumption mechanisms, as explained below.

## Launching mutliple consumption mechanisms


In practical sessions 3 and 4 we saw how to activate a consumption mechanism, that we used to make agents consume resources. In the miniproject scene, we extend this with **4 independent *slots* for the consumption mechanim**. This means that you can define four different ways for subtypes to consume other subtypes. These 4 consumption slots are accessible with `controller.consumption.slot_1`, ..., `controller.consumption.slot_4`.


Below we use two of them, `slot_1` and `slot_2` as an example where:

- Preys consume resources
- Predators consume preys

In [ ]:
# Use slot 1 to make preys consume resources
controller.consumption.slot_1.source_subtype = "prey"
controller.consumption.slot_1.target_subtype = "resource"
controller.consumption.slot_1.range = 1
controller.consumption.slot_1.start = True

In [ ]:
# Use slot 2 to make predator consume preys
controller.consumption.slot_2.source_subtype = "predator"
controller.consumption.slot_2.target_subtype = "prey"
controller.consumption.slot_2.range = 1
controller.consumption.slot_2.start = True

You can refer to [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb) for more detailled explanation of how the consumption mechanism operates. The only change here is that you now have to indicate a slot for each separate consumption mechanism you will use. We use `slot_1` and `slot_2` above, and you can also use `slot_3` and `slot_4` if you want. This way you will be able to implement more complex scenarios, with multiple consumption mechanisms.

## Launching mutliple spawning mechanisms

Similarly to the consumption mechanism, the scene provides **4 independent *slots* for the spawning mechanism**. Below we use two of them, `slot_1` and `slot_2` as an example where:

- New resources spawn every 100 time steps
- New preys spawn every 200 time steps

In [ ]:
# Spawn resources
controller.spawn.slot_1.subtype = "resource"
controller.spawn.slot_1.period = 100
controller.spawn.slot_1.start = True

In [ ]:
# Spawn preys
controller.spawn.slot_2.subtype = "prey"
controller.spawn.slot_2.period = 200
controller.spawn.slot_2.start = True

You can refer to [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb) for more detailed explanation of how the spawning mechanism operates. The only change here is that you now have to indicate a slot for each separate spawning mechanism you will use. We use `slot_1` and `slot_2` above, and you can also use `slot_3` and `slot_4` if you want. 

When activated, a spawning slot will spawn entities up to the number of entities with the corresponding subtype we have previously defined. For instance, we assigned the subtype `"resource"` to 24 entities earlier in this notebook. This means that if all the 24 resource objects are already present on the map, no spawning of resources will occur until at least one resource is consumed. 

In the code cell below, you can try to use slots 3 and 4 of both the consumption and spawn mechanisms, using other subtypes than above, to make sure you understand how this works:

## Avalaible entity attributes

To help you design creative scenarios, we list below all the entity attributes you can play with in your miniproject. You can freely customize the attributes of the entities to you own needs. Remember that you can also dynamically change the attribute of entities during the simulation run with the *routine* mechanism, similarly to [session 4](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_4.ipynb) where we were dynamically modifying the diameters of the agents according to their respective energy levels.

### Available atrribute of all entities (both agents and objects)

- **x_position**: The horizontal position of the entity. Valid values are between 0 and 100 (the full map has a size of 100 by 100).

- **y_position**: The vertical position of the entity. Valid values are between 0 and 100.

- **subtype**: The subtype of the entity. Valid values are the custom subtype labels you have defined at the start of the notebook.

- **exists**: Whether the entity exists or not. By default, all entities are existing. Non-existing entities are not visible on the map and do not interact with other entities. Valid value are `True` (existing) or `False`(not existing).

- **diameter**: The diameter of the entity, i.e. its size. Valid values are strictly positive (>0).

- **color**: The color of the entity. Valid values are [any color from this list](https://docs.bokeh.org/en/latest/docs/reference/colors.html#bokeh-colors-groups).

- **friction**: How much the entity has friction with the "floor". Lower values will make the entity "drifts" more when it collides with other entities, while higher value will make the entity stays more in place. Valid values are strictly positive (>0). Default is 1.

- **mass**: The mass of the entity. Higher values will make the entity harder to move when colliding with other entities, but will also result in more inertia (if it starts drifting, it will drift for longer). Valid values are strictly positive (>0). Default is 1.

- **visible**: Whether the entity is visible on the map or not. The difference with the `exists` attribute is that entities that exist but are not visible still interact with other entities (e.g. they can collide with other entities and the agents can sense them). Valid value are `True` (visible) or `False`(not visible). Default is `True`.


### Attributes only available for agents

- **orientation**: The front direction of the agent (indicated by the small line on it). It is expressed in radians. At 0, the agent will face towards the right side of the map. At $\pi$ (approximately 3.14), it will face to the left side of the map. Valid values are normally between 0 and $2\pi$ (approximately 6.28), although any positive or negative value is also valid (the resulting orientation will be modulo $2\pi$).

- **max_speed**: The maximum speed of the agent when the activation value of both wheels is 1. Valid values are normally positive (>=0), although negative values are also valid (the agent will move backward then). If the speed is too high the agent might miss collisions with other entities and might be harder to control (e.g. oscillating more with an obstacle avoidance behavior).

- **wheel_diameter**: The diameter of the agent's wheels. Changing the wheels diameter mostly have the same effect as changing the maximum speed and we recommend to tune agent's speed with `max_speed` instead. The only potential interest of changing the wheel diameter is to control how sharply the agent will turn (not tested thought). Valid values are strictly positive (>0).

- **proxs_dist_max**: The distance range of the agent proximeter field of view, as explained at the end of [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb). Valid values are strictly positive (>0).

- **proxs_cos_min**: How large the field of view will be, as explained at the end of [session 3](https://github.com/flowersteam/vivarium/blob/upf2026/notebooks/sessions/session_3.ipynb). Valid values are between -1 (360° field of view) and 1 (no field of view). Default value is 0.5 (semi-disk field of view).

- **visible_wheels**: Whether the agent's wheels are visible on the map or not. Valid values are `True` (visible) or 'False` (not visible).

- **visible_proxs**: : Whether the agent's proximeter field of view is visible on the map or not. Valid values are `True` (visible) or `False` (not visible).

### Reading and modifying entity attributes

To check the current values of the entity attributes you can use a `for` loop as we did earlier in this notebook. For instance, let's check the current proximeter distance range of all agents on the map:

In [ ]:
for agent in controller.agents:
    print(agent.proxs_dist_max)

And let's e.g. reduce the proximeter distance range of the predator agents:

In [ ]:
for agent in controller.agents:
    if agent.subtype == "predator":
        agent.proxs_dist_max = 15

We can sometimes observe that the tree object (the largest one) is moving when agents collide with it (if you don't observe it you can try to drag and drop another object on the tree). A tree shouldn't move, right? Maybe its mass is too low or its friction with the floor is too low.  Let's look at the current values of the mass and friction attribute of the tree object:

In [ ]:
for obj in controller.objects:
    if obj.subtype == "tree":
        print(obj.mass, obj.friction)

They are both at their default value (1.0). Let's strongly increase them, such that we have a very massive tree well anchored on the map:

In [ ]:
for obj in controller.objects:
    if obj.subtype == "tree":
        obj.mass = 1000
        obj.friction = 1000         

Now it's a solid tree, you can drag and drop any object on it, it won't move.

## Recap of all the functionalities we have seen during the sessions

In your miniproject, you can use all the functionalities we have seen during the practical sessions. We recommend you use most of them. We provide below a summary of all these functionalities. The content of the practical sessions is [available online](https://github.com/flowersteam/vivarium/tree/upf2026/notebooks/sessions) (only sessions 1 to 4 are relevant, the others are likely outdated).

### Session 1 — First Steps with the Simulator

**Controller Setup**
- `VivariumController.start_session(scene_name="session_1")` — connect to the simulator
- `controller.agents` / `controller.objects` — access entities
- `agent.print_infos()` — display agent attributes

**Entity Attributes**
- `agent.diameter`, `agent.color` — resize and recolor agents
- `object.x_position`, `object.y_position`, `object.color` — reposition and recolor objects
- `agent.x_position`, `agent.y_position`, `agent.orientation` — set position and direction (in radians)

**Motor Control**
- `agent.left_motor`, `agent.right_motor` (0–1) — set wheel speeds
- `agent.max_speed` — adjust maximum speed
- `agent.stop_motors()` — halt both motors

**Sensor Reading**
- `agent.proximeters()` → `(left, right)` — read proximity sensors (0=no detection, 1=contact)

### Session 2 — Implementing Reactive Behaviors

**Behavior Attachment**
- `agent.attach_behavior(fn)` — attach a behavior function
- `agent.detach_behavior(fn)` — detach a specific behavior
- `agent.detach_all_behaviors(stop_motors=True)` — detach all behaviors and stop the motors
- `agent.print_behaviors()` — inspect attached/started behaviors

**Braitenberg Behaviors Implemented**
- `slow_down(agent)` — inhibitory direct connections (slows near obstacles)
- `fear(agent)` — excitatory direct connections (flees away)
- `aggression(agent)` — excitatory crossed connections (charges toward objects)

### Session 3 — Selective Sensing, Parallel Behaviors, Environmental Dynamics

**Subtype Management**
- `controller.subtypes` — list entity subtypes (e.g. `"agent"`, `"resource"`, `"obstacle"`)
- `agent.proximeters(sensed_entities=["resource"])` — filter sensing by subtype

**Proximeter Field of View**
- `agent.proxs_dist_max` — maximum sensing distance
- `agent.proxs_cos_min` (−1 to 1) — field of view angle

**Parallel Behaviors**
- Multiple `attach_behavior()` calls — motor activations averaged across all behaviors
- `agent.print_behaviors()` — inspect all running behaviors

**Additional Behaviors Implemented**
- `shyness(agent)` — crossed inhibitory connections (avoids entities)
- `obstacle_avoidance(agent)` — similar to shyness
- `foraging(agent)` — similar to aggression

**Consumption Mechanism**
- `controller.consumption.source_subtype` — who consumes
- `controller.consumption.target_subtype` — what is consumed
- `controller.consumption.range` — trigger distance
- `controller.consumption.start` — activate/deactivate

**Spawning Mechanism**
- `controller.spawn.subtype` — entity type to spawn
- `controller.spawn.period` — spawn period (how many time steps between spawns)
- `controller.spawn.position_range = [x_min, x_max, y_min, y_max]` — spawn area
- `controller.spawn.start` — activate/deactivate

**Multi-Agent & Entity Management**
- `agent.exists = True/False` — toggle entity existence
- Batch operations via `for agent in controller.agents:`

### Session 4 — Modulating Internal States with Routines

**Behavior Weighting**
- `agent.attach_behavior(fn, weight=value)` — assign a weight (default 1)
- `agent.change_behavior_weight(fn, new_weight=value)` — update weight dynamically
- `agent.print_behaviors(full_infos=True)` — show weights

**Additional Behaviors Implemented**
- `love(agent)` — inhibitory direct connections (stays near agents)
- All 4 canonical Braitenberg behaviors with `sensed_entities` filtering

**Routine Definition & Attachment**
- `def routine(agent):` — no return value; runs every timestep
- `agent.attach_routine(fn)` — attach routine
- `agent.detach_routine(fn)` / `agent.detach_all_routines()` — detach routines
- `agent.print_routines()` — inspect routines

**Internal State**
- `agent.internal.custom_attr = value` — initialize custom state (must be done before attaching routines)
- `agent.internal.custom_attr` — read state inside routines

**Consumption Tracking**
- `agent.has_consumed()` — returns count of consumed entities since last call (resets on each call)

**Key Routines Implemented**
- `energy(agent)` — tracks energy level: decay over time, gain on consumption, clipped to [0, 1]
- `energy_diameter(agent)` — maps energy level to agent diameter
- `foraging_weight(agent)` — dynamically adjusts foraging behavior weight based on energy (low energy → high weight)


## Backing up your work and retrieving previous versions

We strongly recommend to structure the implementation of your scenario in an iterative way. Or in simpler words: first start simple, then add more complexity step by step. The most important is that you can deliver on time a miniproject that works. Therefore, you have interest to first work on a first and easy version of your scenario and save it as a backup once it works, before continuing for the next steps. This way you will ensure that you always have a working version you can deliver, whatever happens next.

In order to save the current version of your notebook, you can click on `File -> Download` in the menu bar at the top of this notebook (the menu bar of the Jupyter notebook, **not** the one of your web browser). Give it a specific name, for example `miniproject_template_V1.ipynb` and save it in the folder indicated when you execute the following cell:

In [ ]:
pwd

Then, if you want to come back to a previous version, first back up your current version as explained just above and stop the Jupyter server with the dedicated red button above this notebook. Then copy-paste the notebook file corresponding to your previous version in the same folder.

Finally, rename the notebook file you have just copied as `miniproject_template.ipynb`. You can then restart the Jupyter server (or just quit and reopen the app as indicated in the Troubleshooting Instructions in Aula Global). This way, the notebook `miniproject_template.ipynb` will always be the version you are currently working on. Alternatively, if you just want to have a quick look at a previous version without having to copy-paste files, you can use the "Open new notebook" blue button above the notebook, navigate to the folder `notebooks/sessions` and open the notebook you want. It will open it in a new browser tab. From there you can e.g. copy-paste parts of the code in your main notebook which is opened in the interface (`miniproject_template.ipynb`).

## Optional extra functionalities

In your miniproject, we recommend to use most of the functionalities that are mentioned above, including those we have seen during the four practical sessions. 

Below we explain other existing functionalities that can also be interesting, but it's up to you to decide if you want to use them or not (no worries if not).

### Calibrating the simulator speed

This session's environment contains more entities than in the previous sessions, which might slow down the simulation.

In case you find the agents are moving too slow, you can increase the number of steps the simulation performs on the server for each step performed in this notebook's controller by mofifying the `controller.simulator.env.num_scan_steps` parameter. It is set to 1 by default. If you double it to 2, your simulation will run approximately twice faster:

In [ ]:
controller.simulator.env.num_scan_steps = 2

Use this mechanism wisely, as increasing this number too high will make your agent's behaviors less reactive, in the sense that the time between the proximeter sensing and the motor activations will be longer. We recommend to not increase it above 8 maximum. Only use integer numbers for this parameter. 

We strongly recommend to calibrate the speed of the simulation **before** you start testing behaviors and routines, as their effects can be highly dependent on the above parameter. This is for instance the case of the `energy` routine we implemented in session 4. Typically, the energy level will decrease slower if `controller.simulator.env.num_scan_steps` is higher.

Another and complementary way to speed up the simulation is to reduce the value of the agent's `friction` parameter. This will make them move faster, although it can also make them drift more.

### Controller routines

In section 4, we have seen how to attach routines to either agents or objects. This is a powerful mechanism to dynamically modulates either existing attributes (e.g. the diameter) or custom internal states (e.g. the energy level). We strongly recommend to use routines in your miniproject (see session 4 for how to use routines).

It is also possible to attach routines to the `controller` itself, enabling to dynamically modulate simulation-wide parameters. With controller routines you can for instance change parameters of the consumption and spawning mechanisms according to specific events in the simulation ; or you can change the attributes of entities according to the value of other entities (e.g. making an object appear whenever another object disappears).

Defining a controller routine is quite similar to defining entity routine: it consists in defining a Python function implementing instructions that will be executed at each simulation time step. The only difference is that a controller routing takes the `controller` as an argument, whereas entity routines takes an entity (either an agent or an object) as an argument. 

As an example, we show below how controller routines can be used to implement a reproduction mechanism. 

#### Agent reproduction

Let's consider the following scenario:
- Prey agents modulate their internal energy levels according to the resources they consume (as in session 4).
- Whenever their energy level is above a certain threshold, they produce an offspring agent next to them.

To implement this, let's first equip the prey agents with an internal energy level modulated by resource consumption, similarly to what we did in session 4:

In [ ]:
# Initialize the energy level of all prey agents at 0.5
for agent in controller.agents:
    if agent.subtype == "prey":
        agent.internal.energy_level = 0.5

# Define a routine updating the energy level of an agent at every time step
# decreasing it at each time step and increasing it when the agent consumes a resource
# (same as in session 4)
def energy(agent): 
    # This function will be executed at each time step
    # on each agent the routine is attached to
    
    # Decrease the agent's energy level by a small amount
    agent.internal.energy_level = agent.internal.energy_level - 0.0002

    # Read the number of resources the agent has consumed since the last time step
    number_of_resources_consumed = agent.has_consumed()

    # Increase the energy level 
    # proportionally to the number of ressources the agents has consumed
    agent.internal.energy_level = agent.internal.energy_level + 0.1 * number_of_resources_consumed

    # Clip the energy level at a minimum value of 0
    if agent.internal.energy_level < 0.0:
        agent.internal.energy_level = 0.0
        
    # and at a maximum value of 1        
    if agent.internal.energy_level > 1.0:
        agent.internal.energy_level = 1.0



In [ ]:
# Attach the energy routine to all prey agents
for agent in controller.agents:
    if agent.subtype == "prey":
        agent.attach_routine(energy)
        
# Note that if you attach the routine to a predator agent
# it will raise an error because we haven't initialized
# the energy level of predator agents in the previous cell

Now each prey agent is modulating its own energy level according to the resources it consumes. 

Next we implement a controller routine for reproducing prey agents. A controller routing is very similar to the entity routines we have seen in session 4, except that it takes the controller as an argument instead of an entity. This enables to implement operations involving multiple entities. This is what we need for a reproduction mechanism, since it involves at least two agents: the parent agent that will reproduce and the offspring agent that will be born.

In [ ]:
def prey_reproduction(controller):   
    
    for agent in controller.agents: # Iterate over all agents       
        
        # If the agent is a prey and its energy level is superior or equal to 0.9,
        # we try to reproduce it
        if agent.subtype == "prey" and agent.internal.energy_level >= 0.9:
                
            # First we list all prey agents that do not currently exist
            # Producing an offspring will correspond to make a non-existing agent existing
            non_existing = [a for a in controller.agents if a.subtype == "prey" and not a.exists]
            
            # If the length of this list is 0,
            # it means that all prey agents currently exist.
            # Therefore there is no possibility to produce a new offspring
            # And we just skip exit the routine for this time step
            if len(non_existing) == 0:
                return  # Exit the routine for this time step
            
            # The instructions below will only be executed if there is at least one non-existing prey agent,
            # (if not, we would have already exited the routine for this time step with the return instruction above)            
            
            # We select the first non-existing prey agent in the list of non-existing prey agents
            offspring_agent = non_existing[0]
            
            # We make it exist (it will appear on the map at the next time step)
            offspring_agent.exists = True
            
            # We place it next to its parent (the agent that is reproducing)
            offspring_agent.x_position = agent.x_position + 1
            offspring_agent.y_position = agent.y_position + 1
            
            # We initialize the energy level of the offspring at 0.5
            offspring_agent.internal.energy_level = 0.5
            
            # Finally, we decrease the energy level of the parent by by the same amount
            agent.internal.energy_level -= 0.5
            
            # Note that the offspring agent was necessarily initialized as a prey agent earlier in this notebook
            # because we selected it among the non-existing *prey* agents
            # Therefore it already has the attributes, behaviors and routines we assigned to all prey agents
            # And we behave as such.
            

Now we can attach the above routine to the controller. It is similar to attaching a routine to an entity as we did in session 4, except that a controller routine is attached to the `controller`:

In [ ]:
controller.attach_routine(prey_reproduction)

Now, each time the energy level of a prey agent will be above 0.9 and the maximum number of preys is not reached (i.e. there are some non-existing preys), the prey will produce an offspring next to it. 

This might be hard to observe in the current simulation because resources are quite sparse, so it might be rare that prey agents manage to raise their energy levels above 0.9. To make it more likely to happen, we can make resources more abundant by spawning them at each time step. For spawning resources, we used `slot_1` of the spawning mechanism, which we can check with:

In [ ]:
controller.spawn.slot_1.subtype

Let's check every how many time steps resources are currently spawn:

In [ ]:
controller.spawn.slot_1.period

Let's spawn them every time step instead to make resources more abundant:

In [ ]:
controller.spawn.slot_1.period = 1

Now you should observe prey agents reproducing from time to time (when they managed to consume several resources in a short time).

### Logging and plotting data from the simulation

All entities and the controller are equipped with a logging mechanism, enabling to record data from the simulation. Let's try it with a prey agent:

In [ ]:
# Prey agents start at index 4 (see earlier in this notebook)
# Here we create a variable called prey_agent
# corresponding to the first prey agent in the list
prey_agent = controller.agents[4]

#### Logging data

Now, let's record some data on this agent. Recording data is realized by the `add_log` method of the agent's `logger`, which requires two arguments: an arbitrary label describing the recorded data, what we call a *topic*, and the data to be recorded. For example:

In [ ]:
# Record the number 1 in a topic called 'test' on prey_agent's logger
prey_agent.logger.add("test", 1)

This stores the data `1` in a topic that we arbitrarily call `"test"`. We can retrieve this data by using the `get_log` function , which requires as argument the name of the topic (`"test"` in this example):

In [ ]:
prey_agent.logger.get("test")

Calling `prey_agent.get_log("test")` returns the list of the data recorded in the topic `"test"` by `prey_agent`. Here it prints `[1]`, a list containing the only data we have stored so far.

Let's add another data to the same topic:

In [ ]:
prey_agent.logger.add("test", 42)

And retrieve the data recorded on this topic:

In [ ]:
prey_agent.logger.get("test")

The second value we have added (`42`), has been appended to the list, which now contains the two recorded data.

We can add another value to another topic:

In [ ]:
prey_agent.logger.add("another_topic", 18)

and retrieve it using `get_log`, this time with the name of this new topic:

In [ ]:
prey_agent.logger.get("another_topic")

Of course the data we previously recorded in the topic `"test"` is still accessible:

In [ ]:
prey_agent.logger.get("test")

The names chosen for the topics are completely arbitrary. They are just labels that you choose for organizing the recorded data according to their meaning. The type of data recorded in a topic is also arbitrary: above we recorded integer values, but we could instead record strings or whatever.

These two functions allow to record various data from the simulation, organizing them by topics differentiated by their names and attaching them to specific agents. Coupled with an appropriate routine running on the agent that continuously calls the `add_log` function, for example to record the agent's energy level through time, this can then be used for generating figures plotting what is happening in the simulation.

Let's define a routine that record the energy level of an agent at each time step:

In [ ]:
def log_energy(agent):
    agent.logger.add("energy_level", agent.internal.energy_level)

And attach it to `prey_agent`:

In [ ]:
prey_agent.attach_routine(log_energy)

We can then access the data recorded on the `"energy_level" topic with:

In [ ]:
prey_agent.logger.get("energy_level")

The amount of data recorder by such a routine can grow pretty quickly though, as it will record a new data every time step (depending on the performance of your computer, this can be several hundreds per second). We can check how many data points have been recorded on a topic so far:

In [ ]:
# the `len` function returns the number of elements in a list
len(prey_agent.logger.get("energy_level"))

To remove all data recorded in a given topic:

In [ ]:
prey_agent.logger.clear("energy_level")

After having clear the data, the topic will continue to record data as long as the corresponding routine is still executed (if you re-execute the last-but-one cell above, you will see it is still recording data).

We can stop the logging routine as any other routine (see session 4):

In [ ]:
prey_agent.detach_routine(log_energy)

And clear all recorded data in the corresponding topic:

In [ ]:
prey_agent.logger.clear("energy_level")

Now the `"energy_level"` topic is cleared and no longer record:

In [ ]:
len(prey_agent.logger.get("energy_level"))

If we want to limit the amount of recorded data, we can instead call the routine every 100 time step by setting the `interval` argument of the `attach_routine` method:

In [ ]:
# First we detach the currently attached routine:
prey_agent.detach_routine(log_energy)

# And reattach it with interval=100
# This means that the energy level will now be logged every 100 time steps instead of every time step
prey_agent.attach_routine(log_energy, interval=100)

#### Plotting data

We can then plot the data recorded on a topic. For this we use the standard Python library for producing plots: `matplotlib`. We first need to import the library on inform the notebook to produce plots directly in the current document:

In [ ]:
# Import the matplotlib library for plotting
import matplotlib.pyplot as plt

# The line below is mandatory to inform the notebook we want to plot directly in it
# (otherwise it will plot in a separate window)
%matplotlib inline

Now we can use `matplotlib` to plot the recorded data. We provide below template code for doing it. If you want to know about how to use matplotlib, see e.g. [this tutorial](https://www.w3schools.com/python/matplotlib_pyplot.asp) (click Next at the end of each page to go to the next one).

Let's plot the recorded energy level of `prey` agent against time:

In [ ]:
# Plot the energy levels recorded by `prey_agent`
plt.plot(prey_agent.logger.get("energy_level"))

# Label the x-axis as "Time"
plt.xlabel("Time")

# Label the y-axis as "Energy level"
plt.ylabel("Energy level")

# Add a title to the plot
plt.title("Plot of energy level against time")

# Display the plot
plt.show()

How about plotting the agent's position through time to better observe its trajectory? For doing this we first need to record the agent position with a new routine:

In [ ]:
def log_position(agent):
    # Record the x and y position of the agent at each time step in two separate topics
    agent.logger.add("x_position", agent.x_position)
    agent.logger.add("y_position", agent.y_position)

Then attach this new routine to our `prey_agent`:

In [ ]:
prey_agent.attach_routine(log_position, interval=10)

Now the agent is recording it's x and y position every 10 time steps.

Wait e.g. 10 second for data to recorded. Then we can plot the trajectory the agent went through since we attached the routine with:

In [ ]:
# Plot the `prey_agent` x position against its y position using the data recorded in the logger
# Here we use '.' as the last argument to the plot function
# to specify that we want to plot points (instead of lines as above)
plt.plot(prey_agent.logger.get("x_position"), prey_agent.logger.get("y_position"), '.')

# Label the axes
plt.xlabel("X")
plt.ylabel("Y")

# Set the limits of the axes to be between 0 and 100 (i.e. the size of the map)
plt.xlim(0, 100)
plt.ylim(0, 100)

# Add a title to the plot
plt.title("Agent positions through time")

# And finally display the plot
plt.show()

We can also plot the data from multiple agents on the same plot. Since we have already defined the routine for recording (x, y) positions, we can just attach it to another prey agent:

In [ ]:
# Create a variable for the next prey agent in the list
prey_agent_2 = controller.agents[5]

# Attach the routine logging the position to this new prey agent
prey_agent_2.attach_routine(log_position)

Wait a bit for position data to be recorded on this new prey agent. Then we can plot the data of both agent's positions on the same plot:

In [ ]:
# Plot the position trajectory of `prey_agent`
plt.plot(prey_agent.logger.get("x_position"), prey_agent.logger.get("y_position"), '.')

# Plot the position trajectory of `prey_agent_2` on the same plot
plt.plot(prey_agent_2.logger.get("x_position"), prey_agent_2.logger.get("y_position"), '.')

# Add a legend to differentiate the two trajectories on the plot
plt.legend(["prey_agent", "prey_agent_2"])

# Label the axes
plt.xlabel("X")
plt.ylabel("Y")

# Set the limits of the axes to be between 0 and 100 (i.e. the size of the map)
plt.xlim(0, 100)
plt.ylim(0, 100)

# Add a title to the plot
plt.title("Agent positions through time")

# And finally display the plot
plt.show()

#### Recording an plotting simulation-wide data

Above we saw how to record and plot data recorded from specific entities. But it might also be interesting to record and plot simulation-wide measures, e.g. how the number of entities of different subtypes evolve through time. Let's for instance define a routine that logs the number of existing preys and resources:

In [ ]:
def log_number_of_preys_and_resources(controller):
    number_of_preys = len([a for a in controller.agents if a.subtype == "prey" and a.exists])
    number_of_resources = len([o for o in controller.objects if o.subtype == "resource" and o.exists])
    controller.logger.add("number_of_preys", number_of_preys)
    controller.logger.add("number_of_resources", number_of_resources)

And attach it to the controller, with an interval of 10 (it will record data every 10 time steps)

In [ ]:
controller.attach_routine(log_number_of_preys_and_resources, interval=10)

Finally, let's plot the number of preys and resources through time:

In [ ]:
plt.plot(controller.logger.get("number_of_preys"))
plt.plot(controller.logger.get("number_of_resources"))

plt.legend(["Number of preys", "Number of resources"])

plt.xlabel("Time")
plt.ylabel("Total number")
plt.title("Number of preys and resources against time")
plt.show()

As observed in the produced plot above, the number of resources is always close to the maximum number. This is because we previously made resources spawn every time step. You can try to reduce the period at which resources will spawn and produce the plot again to observe the consequences.